In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

In [2]:
# region_list = [
#     "Groningen en NO-Drenthe", "Noord-Westelijke Delta", "Overijsselse Vecht",
#     "Limburg", "Vallei en Veluwe", "Achterhoek", "Brabantse Delta",
#     "Friesland", "ARK-NZK", "Noord-Brabant Oost", "Rivierenland",
#     "Scheldestromen", "Zuiderzeeland"
# ]
 
 
# for region in region_list:
#     root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
#     gpkg_path = root_dir / "Aggregated_schakels.gpkg"
#     roads_ex = gpd.read_file(gpkg_path)
 

In [ ]:
#fixing the damages:


import geopandas as gpd
from pathlib import Path

# Read the input file
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
gdf_src = gpd.read_file(input_path)

# Dissolve by 'NET' column with custom aggregation
gdf_dissolved = gdf_src.dissolve(
    by="NET",
    aggfunc={
        "total_length": "sum",
        "flooded_length": "sum",
        "bridge_length_sum": "sum",
        "tunnel_length_sum": "sum",
        "total_damage": "sum",
        "Areas_name": lambda x: ', '.join(x.unique())  # Combine unique values
    }
)

# Reset index if needed
gdf_dissolved = gdf_dissolved.reset_index()

# Save to a new file
output_path = input_path.parent / "Damages_Aggregated_by_Schakels.gpkg"
gdf_dissolved.to_file(output_path, driver="GPKG")

print(f"Dissolved file saved to: {output_path}")


Dissolved file saved to: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Aggregated_by_Schakels.gpkg


In [6]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import re

# Input
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Aggregated_by_Schakels.gpkg")
gdf_src = gpd.read_file(input_path)

# Split Areas_name into individual entries
def split_areas(val):
    if pd.isna(val):
        return []
    parts = [p.strip() for p in re.split(r"[;,\u061B]", str(val))]  # split on ; , and Arabic semicolon
    return [p for p in parts if p]

if "Areas_name" not in gdf_src.columns:
    raise KeyError("Column 'Areas_name' not found.")

gdf_exp = (
    gdf_src.assign(Area=gdf_src["Areas_name"].apply(split_areas))
           .explode("Area", ignore_index=True)
)
gdf_exp = gdf_exp[gdf_exp["Area"].notna() & (gdf_exp["Area"] != "")]

# Calculate metrics
gdf_exp["dam_per_m"] = (
    (gdf_exp["total_damage"] / gdf_exp["total_length"])
    .where(gdf_exp["total_length"].notna() & (gdf_exp["total_length"] != 0))
)

gdf_exp["fraction_flooded"] = (
    (gdf_exp["flooded_length"] / gdf_exp["total_length"])
    .where(gdf_exp["total_length"].notna() & (gdf_exp["total_length"] != 0))
)

# Ranking
gdf_exp["rank_frfl"] = gdf_exp.groupby("Area")["fraction_flooded"].rank(ascending=False, method="min")
gdf_exp["rk_d_m"] = gdf_exp.groupby("Area")["dam_per_m"].rank(ascending=False, method="min")

# Prepare ranked output
required = ["Area", "rank_frfl", "rk_d_m", "dam_per_m", "NET", "total_damage", "fraction_flooded",
            "total_length", "flooded_length", "tunnel_length_sum", "bridge_length_sum", "geometry"]
missing = [c for c in required if c not in gdf_exp.columns]
if missing:
    raise KeyError(f"Missing in gdf_exp: {missing}")

gdf_ranked = (
    gdf_exp.loc[:, required]
            .sort_values(["Area", "rank_frfl", "rk_d_m", "dam_per_m", "fraction_flooded"],
                         ascending=[True, True, True, False, False], kind="mergesort")
            .reset_index(drop=True)
)

# Output paths
base_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
base_dir.mkdir(parents=True, exist_ok=True)
gpkg_path = base_dir / "Damage_risk_ranking_schakels.gpkg"
shp_path  = base_dir / "Damage_risk_ranking_schakels.shp"

# Save GPKG
gdf_ranked.to_file(gpkg_path, layer="ranked_rows", driver="GPKG")

# Save SHP (shorten field names)
gdf_ranked_shp = gdf_ranked.rename(columns={
    "total_damage": "tot_dam",
    "fraction_flooded": "frac_fld",
    "rk_d_m": "rk_d_m",
    "rank_frfl": "rk_frfl"
}).copy()

# Convert ranks to integer for SHP
gdf_ranked_shp["rk_d_m"] = gdf_ranked_shp["rk_d_m"].astype("Int64")
gdf_ranked_shp["rk_frfl"] = gdf_ranked_shp["rk_frfl"].astype("Int64")

gdf_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")

print(f"Wrote:\n- {gpkg_path}\n- {shp_path}")


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_21580\4039187361.py:76: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")


Wrote:
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damage_risk_ranking_schakels.gpkg
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damage_risk_ranking_schakels.shp


In [1]:
# considering the country level damages

import geopandas as gpd
import pandas as pd
from pathlib import Path

# Load losses ranking (country-wide)
losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp")
losses_rank = gpd.read_file(losses_rank_path, driver="ESRI Shapefile")

# Paths for (country-wide merged) damages
root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
gpkg_path = root_dir / "Damage_risk_ranking_ver04.gpkg"

# Read damages
damages_rank = gpd.read_file(gpkg_path, driver="GPKG")

# Optional: damage per meter
damages_rank["dam_per_m"] = (
    (damages_rank["total_damage"] / damages_rank["total_length"])
    .where(damages_rank["total_length"].notna() & (damages_rank["total_length"] != 0))
)

# Merge VHLH and damages (country-wide), keep Area for groupwise ranking
vhlh_df = losses_rank[['NETWERKSCH', 'rk_VHLH', 'Area', 'Areas_name']].copy()
damage_df = damages_rank[['NETWERKSCH_HWN', 'rk_d_m', 'flooded_length', 'tunnel_length_sum', 'geometry']].copy()

df = pd.merge(damage_df, vhlh_df, left_on='NETWERKSCH_HWN', right_on='NETWERKSCH', how='inner')

# Fill missing tunnel lengths with 0
df['tunnel_length_sum'] = df['tunnel_length_sum'].fillna(0)

# Convert back to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=damages_rank.crs)

# ✅ Invert ranks for VHLH and Damage per Area
max_vhlh_by_area = gdf.groupby('Area')['rk_VHLH'].transform('max')
max_damage_by_area = gdf.groupby('Area')['rk_d_m'].transform('max')
gdf['inv_VHLH'] = (max_vhlh_by_area + 1) - gdf['rk_VHLH']
gdf['inv_Damage'] = (max_damage_by_area + 1) - gdf['rk_d_m']

# ✅ Rank tunnels within Area and invert
gdf['tunnel_rank'] = gdf.groupby('Area')['tunnel_length_sum'].rank(method='dense', ascending=False)
max_tunnel_by_area = gdf.groupby('Area')['tunnel_rank'].transform('max')
gdf['inv_Tunnel'] = (max_tunnel_by_area + 1) - gdf['tunnel_rank']

# ✅ Rank flooded_length within Area
gdf['flooded_rank'] = gdf.groupby('Area')['flooded_length'].rank(method='dense', ascending=False)

# ✅ Compute weighted score (Damage, VHLH, Tunnel)
weights = {'Damage': 0.5, 'VHLH': 0.3, 'Tunnels': 0.2}
gdf['Total'] = (
    gdf['inv_VHLH'] * weights['VHLH'] +
    gdf['inv_Damage'] * weights['Damage'] +
    gdf['inv_Tunnel'] * weights['Tunnels']
)

# Sort and assign final rank per Area
gdf = gdf.sort_values(['Area', 'Total'], ascending=[True, False])
gdf['Final_rank'] = gdf.groupby('Area')['Total'].rank(method='first', ascending=False).astype(int)

# ✅ Keep flooded_length and flooded_rank in final output
columns_to_keep = [
    'NETWERKSCH', 'Area', 'Areas_name', 'rk_VHLH', 'rk_d_m',
    'flooded_length', 'flooded_rank', 'tunnel_length_sum',
    'inv_VHLH', 'inv_Damage', 'inv_Tunnel', 'Total', 'Final_rank', 'geometry'
]
gdf = gdf[columns_to_keep]

# Save output
out_path = root_dir / "Combined_Risk_Ranking_country_scale_ver02.gpkg"
gdf.to_file(out_path, driver="GPKG")

print(f"Saved ranking to {out_path}")

Saved ranking to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Combined_Risk_Ranking_country_scale_ver02.gpkg


In [6]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Regions list
region_list = [
    "Groningen en NO-Drenthe", "Noord-Westelijke Delta", "Overijsselse Vecht",
    "Limburg", "Vallei en Veluwe", "Achterhoek", "Brabantse Delta",
    "Friesland", "ARK-NZK", "Noord-Brabant Oost", "Rivierenland",
    "Scheldestromen", "Zuiderzeeland"
]

# Load losses ranking (country-wide)
losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp")
losses_rank = gpd.read_file(losses_rank_path, driver="ESRI Shapefile")

for region in region_list:
    print(f"Processing region: {region}")
    
    # Paths for region-specific damages
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    gpkg_path = root_dir / "ranked_Aggregated_schakels.gpkg"
    
    # Read damages for this region
    damages_rank = gpd.read_file(gpkg_path, driver="GPKG")
    
    # Filter losses for this region
    losses_rank_region = losses_rank[losses_rank["Area"] == region]
    
    # Merge VHLH and damages
    vhlh_df = losses_rank_region[['NETWERKSCH', 'rk_VHLH', 'Area', 'Areas_name']].copy()
    damage_df = damages_rank[['NETWERKSCH_HWN', 'rk_d_m', 'flooded_length', 'tunnel_length_sum', 'geometry']].copy()
    
    df = pd.merge(damage_df, vhlh_df, left_on='NETWERKSCH_HWN', right_on='NETWERKSCH', how='inner')
    
    # Fill missing tunnel lengths with 0
    df['tunnel_length_sum'] = df['tunnel_length_sum'].fillna(0)
    
    # Convert back to GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=damages_rank.crs)
    
    # ✅ Invert ranks for VHLH and Damage
    max_vhlh = gdf['rk_VHLH'].max()
    max_damage = gdf['rk_d_m'].max()
    gdf['inv_VHLH'] = (max_vhlh + 1) - gdf['rk_VHLH']
    gdf['inv_Damage'] = (max_damage + 1) - gdf['rk_d_m']
    
    # ✅ Rank tunnels within region and invert
    gdf['tunnel_rank'] = gdf['tunnel_length_sum'].rank(method='dense', ascending=False)
    max_tunnel = gdf['tunnel_rank'].max()
    gdf['inv_Tunnel'] = (max_tunnel + 1) - gdf['tunnel_rank']
    
    # ✅ Rank flooded_length (no weighting yet)
    gdf['flooded_rank'] = gdf['flooded_length'].rank(method='dense', ascending=False)
    
    # ✅ Compute weighted score (Damage, VHLH, Tunnel)
    weights = {'Damage': 0.5, 'VHLH': 0.3, 'Tunnels': 0.2}
    gdf['Total'] = (gdf['inv_VHLH'] * weights['VHLH'] +
                    gdf['inv_Damage'] * weights['Damage'] +
                    gdf['inv_Tunnel'] * weights['Tunnels'])
    
    # Sort and assign final rank
    gdf = gdf.sort_values('Total', ascending=False)
    gdf['Final_rank'] = range(1, len(gdf)+1)
    
    # ✅ Keep flooded_length and flooded_rank in final output
    columns_to_keep = ['NETWERKSCH', 'rk_VHLH', 'rk_d_m', 'flooded_length', 'flooded_rank',
                       'tunnel_length_sum', 'inv_VHLH', 'inv_Damage', 'inv_Tunnel',
                       'Total', 'Final_rank', 'geometry']
    gdf = gdf[columns_to_keep]
    
    # Save output for this region
    out_path = root_dir / "Combined_Risk_Ranking.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    
    print(f"Saved ranking for {region} to {out_path}")

Processing region: Groningen en NO-Drenthe
Saved ranking for Groningen en NO-Drenthe to P:\bovenregionale-stresstest-hwn\Analysis\Groningen en NO-Drenthe\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Noord-Westelijke Delta


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking')) failed: disk I/O error"


Saved ranking for Noord-Westelijke Delta to P:\bovenregionale-stresstest-hwn\Analysis\Noord-Westelijke Delta\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Overijsselse Vecht


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 181077.28359, min_y = 461448.9661, max_x = 268287.1529999999, max_y = 580513.260155 WHERE lower(table_name) = lower('Combined_Risk_Ranking') AND Lower(data_type) = 'features') failed: disk I/O error"

...

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking')) failed: unable to open database file"


Saved ranking for Friesland to P:\bovenregionale-stresstest-hwn\Analysis\Friesland\Outputs\Combined_Risk_Ranking.gpkg
Processing region: ARK-NZK
Saved ranking for ARK-NZK to P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Noord-Brabant Oost
Saved ranking for Noord-Brabant Oost to P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Rivierenland
Saved ranking for Rivierenland to P:\bovenregionale-stresstest-hwn\Analysis\Rivierenland\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Scheldestromen
Saved ranking for Scheldestromen to P:\bovenregionale-stresstest-hwn\Analysis\Scheldestromen\Outputs\Combined_Risk_Ranking.gpkg
Processing region: Zuiderzeeland


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking')) failed: disk I/O error"


Saved ranking for Zuiderzeeland to P:\bovenregionale-stresstest-hwn\Analysis\Zuiderzeeland\Outputs\Combined_Risk_Ranking.gpkg


In [4]:
losses_rank

,NETWERKSCH,VHLH_AL_E_,VHLH_L1_E_,VHLH_L2_E_,VHLH_L3_E_,F_EV2_ma,F_EV1_me,VRPC_E_WR,AL_E_WR,L2+L3_E_WR,...,total_getr,total_ge_1,VOT_L1_gem,VOT_L2L3_g,VOT_total,Areas,Areas_name,Area,rk_VHLH,geometry
0,004-0050-L,4708226.0,4284462.0,188302.0,235394.0,1638.0,3.535830,9.0,68985.0,6208.0,...,423696.0,4785744.0,44644094.04,19769655.36,64413749.40,1,ARK-NZK,ARK-NZK,1,"MULTILINESTRING ((95469.014 462283.593, 95503...."
1,004-0050-R,4694558.0,4225137.0,234745.0,234745.0,1658.0,3.385062,10.0,67955.0,6796.0,...,469490.0,4719478.0,44025927.54,21906403.40,65932330.94,1,ARK-NZK,ARK-NZK,2,"MULTILINESTRING ((97245.105 463377.551, 96719...."
2,027-0060-R,3131826.0,2658111.0,273354.0,200332.0,696.0,4.046029,15.0,107994.0,16334.0,...,473686.0,2969110.0,27697516.62,22102188.76,49799705.38,1,ARK-NZK,ARK-NZK,3,"MULTILINESTRING ((138212.609 451088.321, 13824..."
3,027-0060-L,2859400.0,2423385.0,251285.0,184701.0,696.0,4.075556,15.0,98600.0,15034.0,...,435986.0,2706921.0,25251671.70,20343106.76,45594778.46,1,ARK-NZK,ARK-NZK,4,"MULTILINESTRING ((139559.419 455700.880, 13956..."
4,009-0020-L,1912549.0,1723304.0,105194.0,84051.0,698.0,2.484615,10.0,65761.0,6507.0,...,189245.0,1924930.0,17956827.68,8830171.70,26786999.38,1,ARK-NZK,ARK-NZK,5,"MULTILINESTRING ((113074.158 482422.832, 11309..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
585,006-0080-R,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,NaN,NaN,0.00,0.00,0.00,1,Zuiderzeeland,Zuiderzeeland,8,"MULTILINESTRING ((179908.647 533696.606, 17989..."
586,006-1010-L,0.0,0.0,0.0,0.0,0.0,0.000000,20.0,3600.0,705.0,...,NaN,NaN,0.00,0.00,0.00,1,Friesland;Zuiderzeeland,Zuiderzeeland,8,"MULTILINESTRING ((177524.515 540654.591, 17750..."
587,006-1010-R,0.0,0.0,0.0,0.0,0.0,0.000000,21.0,21900.0,4685.0,...,NaN,NaN,0.00,0.00,0.00,1,Friesland;Zuiderzeeland,Zuiderzeeland,8,"MULTILINESTRING ((177647.152 539881.019, 17761..."
588,050-0110-L,0.0,0.0,0.0,0.0,0.0,0.000000,18.0,8020.0,1460.0,...,NaN,NaN,0.00,0.00,0.00,1,Overijsselse Vecht;Zuiderzeeland,Zuiderzeeland,8,"MULTILINESTRING ((181077.284 523452.041, 18108..."
